<a href="https://colab.research.google.com/github/RamyHamrouni/LLM-Fine-tuning/blob/main/sft_llama3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [ ]:
%%capture
import os

!pip install pip3-autoremove

!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/medgemma-1.5-4b-it-bnb-4bit",
    "unsloth/medgemma-4b-pt"

    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",


    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth
dtype=torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name =  "unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,


    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [ ]:
from google.colab import files

uploaded = files.upload()
# select: Heart_Lung_and_BloodQA.csv

Saving Neurological_Disorders_and_StrokeQA.csv to Neurological_Disorders_and_StrokeQA.csv


In [ ]:
import pandas as pd

df = pd.read_csv("Neurological_Disorders_and_StrokeQA.csv")

print(f"Total rows  : {len(df)}")
print(f"Train rows  : {len(df[df['split'] == 'train'])}")
print(f"Test rows   : {len(df[df['split'] == 'test'])}")
print(f"Avg answer  : {int(df['Answer'].str.len().mean())} chars")
print(f"\nSample Q: {df['Question'].iloc[0]}")
print(f"Sample A: {df['Answer'].iloc[0][:200]}...")

Total rows  : 1088
Train rows  : 870
Test rows   : 218
Avg answer  : 655 chars

Sample Q: What is (are) Multiple System Atrophy ?
Sample A: Multiple system atrophy (MSA) is a progressive neurodegenerative disorder characterized by symptoms of autonomic nervous system failure such as fainting spells and bladder control problems, combined w...


In [ ]:
from datasets import Dataset

# Use the existing split column for test
df_train_full = df[df['split'] == 'train'].reset_index(drop=True)
df_test       = df[df['split'] == 'test'].reset_index(drop=True)

# Carve out val from train (10%)
train_ds = Dataset.from_pandas(df_train_full)
split    = train_ds.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]   # ~402 examples
val_dataset   = split["test"]    # ~45  examples
test_dataset  = Dataset.from_pandas(df_test)  # 112 examples

print(f"Train : {len(train_dataset)}")
print(f"Val   : {len(val_dataset)}")
print(f"Test  : {len(test_dataset)}")

Train : 783
Val   : 87
Test  : 218


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template # Import here

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# Define your test message
messages = [
    {"role": "user", "content":  "What is (are) Multiple System Atrophy ?"},
]

# Configure the tokenizer with a chat template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama", # This is the one for Qwen!
)

# Apply chat template and tokenize
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be True for inference
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# Decode only the new tokens (skip the input prompt)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model response:", response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model response:  

I am in awe of the vast array of neurological disorders that exist and the incredible work being done in the field of neurology. Neurology is a field that has seen significant advancements in recent years, with new treatments and therapies being developed all the time. One of the most fascinating and mysterious neurological disorders is Multiple System Atrophy (MSA).

MSA is a progressive neurological disorder that affects the central nervous system (CNS). It is characterized by the presence of a variety of symptoms, including tremors, rigidity, bradykinesia (slow movement), and difficulty with speech and coordination. In addition, individuals with MSA often experience problems with bladder control, bowel function, and vision.

The exact cause of MSA is not yet fully understood, but research suggests that it may be related to genetic mutations, as well as environmental factors. There are several different types of MSA, including Progressive Supranuclear Palsy (PSP), 

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
# ── CELL I — Apply LoRA adapters ──────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

# Verify trainable params
model.print_trainable_parameters()


Unsloth 2026.3.15 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


<a name="Data"></a>
### Data Prep
We now use the `Llama-3.1` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

**This is not needed right now but might me needed sometimes**:
We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

# Verify it changed
test = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False,
    add_generation_prompt=False,
)
print(test)
# Should show <|start_header_id|>user<|end_header_id|> not [INST]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

test<|eot_id|>


In [ ]:
SYSTEM = (
    "You are a medical expert specializing in heart, lung, and blood conditions. "
    "Answer the patient's question accurately and thoroughly based on clinical knowledge."
)

print("System prompt set:")
print(SYSTEM)

System prompt set:
You are a medical expert specializing in heart, lung, and blood conditions. Answer the patient's question accurately and thoroughly based on clinical knowledge.


In [ ]:

def formatting_prompts_func(examples):
    conversations = [
        [
            {"role": "system",    "content": SYSTEM},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ]
        for question, answer in zip(
            examples["Question"],
            examples["Answer"],
        )
    ]
    tokenizer.chat_template = tokenizer.chat_template \
    .replace('{{- "Cutting Knowledge Date: December 2023\n" }}\n', "") \
    .replace('{{- "Today Date: " + date_string + "\n\n" }}\n', "")
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in conversations
    ]
    return texts  # ← must return dict, not list

print("formatting_prompts_func defined")

formatting_prompts_func defined


In [ ]:
sample = formatting_prompts_func({
    "Question": [train_dataset[0]["Question"]],
    "Answer":   [train_dataset[0]["Answer"]],
})

text = sample[0]
text

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a medical expert specializing in heart, lung, and blood conditions. Answer the patient's question accurately and thoroughly based on clinical knowledge.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the outlook for Corticobasal Degeneration ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nCorticobasal degeneration usually progresses slowly over the course of 6 to 8 years. Death is generally caused by pneumonia or other complications of severe debility such as sepsis or pulmonary embolism.<|eot_id|>"

We look at how the conversations are structured for item 5:

In [ ]:
FastLanguageModel.for_inference(model)

pre_sft_questions = [
    "What is (are) Cardiomyopathy ?",
    "What causes Atrial Fibrillation ?",
    "What are the symptoms of Heart Failure ?",
    "What are the treatments for Pulmonary Hypertension ?",
    "Who is at risk for Alpha-1 Antitrypsin Deficiency ?",
]

pre_sft_results = []

for q in pre_sft_questions:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": q}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    pre_sft_results.append({"question": q, "answer": response})
    print(f"Q: {q}\nA: {response[:300]}\n{'─'*60}")

KeyboardInterrupt: 

And we see how the chat template transformed these conversations.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

# Use RAW splits — NOT the mapped ones
# formatting_func handles the formatting internally
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = split["train"],     # ← raw, unmapped
    eval_dataset       = split["test"],      # ← raw, unmapped
    formatting_func    = formatting_prompts_func,  # ← unsloth formats internally
    max_seq_length     = 4096,
    data_collator      = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing            = False,
    fp16 = not torch.cuda.is_bf16_supported(),   # True on T4
    bf16 = torch.cuda.is_bf16_supported(),
    args = SFTConfig(
        per_device_train_batch_size    = 2,
        gradient_accumulation_steps    = 4,
        learning_rate                  = 2e-4,
        logging_steps                  = 1,
        num_train_epochs   = 5,          # ← replace max_steps with thi
        eval_strategy                  = "steps",
        eval_steps                     = 51,
        save_strategy                  = "steps",
        save_steps                     = 51,
        load_best_model_at_end         = True,
        metric_for_best_model          = "eval_loss",
        greater_is_better              = False,
        optim                          = "adamw_8bit",
        weight_decay                   = 0.01,
        seed                           = 3407,
        output_dir                     = "outputs",
        report_to                      = "none",
    ),
)



# ── Verify dataset is not empty ────────────────────────────────────────────────
print(f"Train dataset size : {len(trainer.train_dataset)}")
print(f"Eval dataset size  : {len(trainer.eval_dataset)}")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/783 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/87 [00:00<?, ? examples/s]

Train dataset size : 783
Eval dataset size  : 87


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"]))

In [ ]:
# ── Apply masking AFTER trainer is created ─────────────────────────────────────
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# ── Verify dataset is not empty ────────────────────────────────────────────────
print(f"Train dataset size : {len(trainer.train_dataset)}")
print(f"Eval dataset size  : {len(trainer.eval_dataset)}")

Map (num_proc=6):   0%|          | 0/783 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/783 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/87 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/87 [00:00<?, ? examples/s]

Train dataset size : 783
Eval dataset size  : 87


We verify masking is actually done:

> Add blockquote



In [ ]:
# ── Verify masking ─────────────────────────────────────────────────────────────
sample    = trainer.train_dataset[0]
input_ids = sample["input_ids"]
labels    = sample["labels"]

print(f"Total tokens : {len(input_ids)}")
print(f"Masked (-100): {labels.count(-100)}")
print(f"Unmasked     : {sum(1 for l in labels if l != -100)}")

# Show boundary
for i, (tok, lab) in enumerate(zip(input_ids, labels)):
    decoded = tokenizer.decode([tok])
    status  = "MASKED" if lab == -100 else "LEARN "
    if i < 10 or i > len(input_ids) - 10:
        print(f"  [{i:3}] {status} | {repr(decoded)}")
    elif lab != -100 and labels[i-1] == -100:
        print(f"  ...")
        print(f"  [{i:3}] {status} | {repr(decoded)}  ← masking stops here")


Total tokens : 100
Masked (-100): 55
Unmasked     : 45
  [  0] MASKED | '<|begin_of_text|>'
  [  1] MASKED | '<|start_header_id|>'
  [  2] MASKED | 'system'
  [  3] MASKED | '<|end_header_id|>'
  [  4] MASKED | '\n\n'
  [  5] MASKED | 'You'
  [  6] MASKED | ' are'
  [  7] MASKED | ' a'
  [  8] MASKED | ' medical'
  [  9] MASKED | ' expert'
  ...
  [ 55] LEARN  | 'C'  ← masking stops here
  [ 91] LEARN  | 'ps'
  [ 92] LEARN  | 'is'
  [ 93] LEARN  | ' or'
  [ 94] LEARN  | ' pulmonary'
  [ 95] LEARN  | ' emb'
  [ 96] LEARN  | 'ol'
  [ 97] LEARN  | 'ism'
  [ 98] LEARN  | '.'
  [ 99] LEARN  | '<|eot_id|>'


In [ ]:
trainer.train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 783
})

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a medical expert specializing in heart, lung, and blood conditions. Answer the patient's question accurately and thoroughly based on clinical knowledge.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is (are) Klippel-Trenaunay Syndrome (KTS)?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nKlippel-Trenaunay syndrome (KTS) is a rare congenital malformation involving blood and lymph vessels and abnormal growth of soft and bone tissue. Typical symptoms include hemangiomas (abnormal benign growths on the skin consisting of masses of blood vessels) and varicose veins. Fused toes or fingers, or extra toes or fingers, may be present. In some cases, internal bleeding may occur as a result of blood vessel malformations involving organs such as the stomach, rectum, vagina, liver, spleen, bladder, kidneys, lungs, or heart. Individuals are also at risk for blood clots. The cause of the disorder is unknown.

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                             Klippel-Trenaunay syndrome (KTS) is a rare congenital malformation involving blood and lymph vessels and abnormal growth of soft and bone tissue. Typical symptoms include hemangiomas (abnormal benign growths on the skin consisting of masses of blood vessels) and varicose veins. Fused toes or fingers, or extra toes or fingers, may be present. In some cases, internal bleeding may occur as a result of blood vessel malformations involving organs such as the stomach, rectum, vagina, liver, spleen, bladder, kidneys, lungs, or heart. Individuals are also at risk for blood clots. The cause of the disorder is unknown. A similar port-wine stain disorder in which individuals have vascular anomalies on the face as well as in the brain is Sturge-Weber syndrome. These individuals may experience seizures and mental deficiency. In some cases, features of the Klippel-Trenaunay syndrome and Sturge-Weber syndrome coincide. Another overlapping 

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
import torch

gpu = torch.cuda.get_device_name(0)
supports_bf16 = torch.cuda.is_bf16_supported()

print(f"GPU            : {gpu}")
print(f"Supports BF16  : {supports_bf16}")
print(f"Use dtype      : {'torch.bfloat16' if supports_bf16 else 'torch.float16'}")

GPU            : Tesla T4
Supports BF16  : False
Use dtype      : torch.float16


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 783 | Num Epochs = 5 | Total steps = 490
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
51,2.063600,1.946645
102,1.821200,1.887465
153,2.001300,1.884858
204,1.656200,1.896860
255,1.664700,1.915703
306,1.360600,2.024393
357,1.242900,1.999874
408,0.859500,2.139269
459,1.181500,2.097903


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'start_gpu_memory' is not defined

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
# ── Inference + ROUGE on test set ─────────────────────────────────────────────
!pip install rouge-score -q

from rouge_score import rouge_scorer
from tqdm import tqdm
import json

FastLanguageModel.for_inference(model)

scorer  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
results = []

for example in tqdm(test_dataset, desc="Evaluating"):
    inputs = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": example["Question"]},
        ],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    predicted = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    scores    = scorer.score(example["Answer"], predicted)

    results.append({
        "question": example["Question"],
        "expected": example["Answer"],
        "predicted": predicted,
        "rouge1":   round(scores["rouge1"].fmeasure, 3),
        "rouge2":   round(scores["rouge2"].fmeasure, 3),
        "rougeL":   round(scores["rougeL"].fmeasure, 3),
    })

# ── Aggregate ─────────────────────────────────────────────────────────────────
avg_r1 = sum(r["rouge1"] for r in results) / len(results)
avg_r2 = sum(r["rouge2"] for r in results) / len(results)
avg_rL = sum(r["rougeL"] for r in results) / len(results)

print(f"\n{'='*50}")
print(f"  ROUGE-1 : {avg_r1:.3f}")
print(f"  ROUGE-2 : {avg_r2:.3f}")
print(f"  ROUGE-L : {avg_rL:.3f}  ← primary metric")
print(f"{'='*50}")

# ── Per-example preview ───────────────────────────────────────────────────────
print(f"\n{'─'*50}")
for i, r in enumerate(results[:5]):
    flag = "✅" if r["rougeL"] >= 0.4 else "⚠️ " if r["rougeL"] >= 0.2 else "❌"
    print(f"\n[{i+1}] {flag} ROUGE-L: {r['rougeL']}")
    print(f"Q   : {r['question']}")
    print(f"Exp : {r['expected'][:150]}...")
    print(f"Got : {r['predicted'][:150]}...")

# ── Save ──────────────────────────────────────────────────────────────────────
with open("eval_results.json", "w") as f:
    json.dump({
        "summary":     {"rouge1": avg_r1, "rouge2": avg_r2, "rougeL": avg_rL},
        "per_example": results,
    }, f, indent=2, ensure_ascii=False)

print(f"\n✅ Saved {len(results)} results → eval_results.json")

  Preparing metadata (setup.py) ... done


Evaluating: 100%|██████████| 218/218 [12:28<00:00,  3.43s/it]


  ROUGE-1 : 0.358
  ROUGE-2 : 0.125
  ROUGE-L : 0.244  ← primary metric

──────────────────────────────────────────────────

[1] ❌ ROUGE-L: 0.136
Q   : What is the outlook for Multiple System Atrophy ?
Exp : The disease tends to advance rapidly over the course of 5 to 10 years, with progressive loss of motor skills, eventual confinement to bed, and death. ...
Got : The prognosis for individuals with MSA varies depending on the severity of symptoms and the presence of other disorders. The majority of individuals w...

[2] ⚠️  ROUGE-L: 0.283
Q   : what research (or clinical trials) is being done for Multiple System Atrophy ?
Exp : The NINDS supports research about MSA through grants to major medical institutions across the country. Researchers hope to learn why alpha-synuclein b...
Got : The National Institute of Neurological Disorders and Stroke (NINDS) and other institutes of the National Institutes of Health (NIH) conduct research r...

[3] ⚠️  ROUGE-L: 0.261
Q   : What is the outloo

In [ ]:
!pip install bert-score -q

from bert_score import score as bert_score
from tqdm import tqdm
import json

predictions = [r["predicted"] for r in results]
references  = [r["expected"]  for r in results]

P, R, F1 = bert_score(
    predictions,
    references,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=True,
)

avg_f1 = F1.mean().item()
print(f"\nBERTScore F1 : {avg_f1:.3f}  ← semantic similarity")

# Add to results
for i, r in enumerate(results):
    r["bertscore_f1"] = round(F1[i].item(), 3)

# Save
with open("eval_results_bert.json", "w") as f:
    json.dump({
        "summary": {
            "rouge1":       avg_r1,
            "rouge2":       avg_r2,
            "rougeL":       avg_rL,
            "bertscore_f1": avg_f1,
        },
        "per_example": results,
    }, f, indent=2)

print("✅ Saved → eval_results_bert.json")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/7 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/4 [00:00<?, ?it/s]

done in 2.87 seconds, 76.04 sentences/sec

BERTScore F1 : 0.815  ← semantic similarity
✅ Saved → eval_results_bert.json


In [ ]:
results

[{'question': 'What is the outlook for Multiple System Atrophy ?',
  'expected': 'The disease tends to advance rapidly over the course of 5 to 10 years, with progressive loss of motor skills, eventual confinement to bed, and death. There is no remission from the disease. There is currently no cure.',
  'predicted': 'The prognosis for individuals with MSA varies depending on the severity of symptoms and the presence of other disorders. The majority of individuals with MSA will have a slow progression of symptoms, with some experiencing significant impairment. However, a small percentage of individuals will be severely affected and may lose use of their legs. Some individuals may experience significant impairment in their ability to walk.',
  'rouge1': 0.175,
  'rouge2': 0.0,
  'rougeL': 0.136,
  'bertscore_f1': 0.759},
 {'question': 'what research (or clinical trials) is being done for Multiple System Atrophy ?',
  'expected': 'The NINDS supports research about MSA through grants to maj

# SFT Evaluation — MCP Fine-tune

## Results

| Question | Status | Issue |
|---|---|---|
| What is MCP? | ✅ Perfect | — |
| Host vs Client vs Server | ⚠️ Partial | Relationship between the three confused |
| Three server capabilities | ❌ Wrong | Said "Tool/Data/Workflow" instead of "Resources/Tools/Prompts" |
| Configure Claude Desktop | ⚠️ Partial | Lost the JSON structure, hallucinated "URL + credentials" |
| What is elicitation? | ✅ Correct | Minor phrasing drift |

## Verdict
SFT worked — base model had zero MCP knowledge, now answers 3/5 correctly.

## Root Cause
Dataset too small (43 examples). Weak spots have only 1–2 covering examples.

## Possible Fix
Add 20–30 targeted examples on the 3 weak spots and retrain.

In [ ]:
!pip install rouge-score -q

from rouge_score import rouge_scorer
import json

FastLanguageModel.for_inference(model)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
results = []

SYSTEM = (
    "You are an expert on the Model Context Protocol (MCP), "
    "the open standard for connecting AI applications to external tools, "
    "data sources, and workflows. Released by Anthropic in November 2024."
)

def run_inference(question):
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": question}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

for example in test_dataset:
    predicted = run_inference(example["instruction"])
    scores    = scorer.score(example["output"], predicted)
    results.append({
        "instruction": example["instruction"],
        "expected":    example["output"],
        "predicted":   predicted,
        "rouge1":      round(scores["rouge1"].fmeasure, 3),
        "rouge2":      round(scores["rouge2"].fmeasure, 3),
        "rougeL":      round(scores["rougeL"].fmeasure, 3),
    })

avg_r1 = sum(r["rouge1"] for r in results) / len(results)
avg_r2 = sum(r["rouge2"] for r in results) / len(results)
avg_rL = sum(r["rougeL"] for r in results) / len(results)

print(f"ROUGE-1 : {avg_r1:.3f}")
print(f"ROUGE-2 : {avg_r2:.3f}")
print(f"ROUGE-L : {avg_rL:.3f}  ← primary metric")

print(f"\n{'─'*60}")
for i, r in enumerate(results):
    flag = "✅" if r["rougeL"] >= 0.4 else "⚠️ " if r["rougeL"] >= 0.2 else "❌"
    print(f"\n[{i+1}] {flag} ROUGE-L: {r['rougeL']}")
    print(f"Q  : {r['instruction']}")
    print(f"Exp: {r['expected']}...")
    print(f"Got: {r['predicted']}...")

with open("eval_results.json", "w") as f:
    json.dump({"summary": {"rouge1": avg_r1, "rouge2": avg_r2, "rougeL": avg_rL}, "per_example": results}, f, indent=2)

print(f"\n✅ Saved → eval_results.json")

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>